# ADSP 32023 — Advanced Computer Vision: Assignment 1

In [ ]:
import os, json, time, hashlib, zipfile, platform, warnings
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Image as DisplayImage
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss
ROOT = Path.cwd()
if (ROOT/'cv_assignment1').is_dir(): ROOT = ROOT/'cv_assignment1'
for name in ['data','results','figures','models','logs','checkpoints']:(ROOT/name).mkdir(exist_ok=True)
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)
keras.utils.set_random_seed(42)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})
def show(name):
    p=ROOT/'figures'/name; plt.savefig(p,bbox_inches='tight'); plt.close(); display(DisplayImage(filename=str(p)))
def dataset(x,y,shuffle=False,batch=64):
    d=tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:d=d.shuffle(len(y),seed=42,reshuffle_each_iteration=True)
    opt=tf.data.Options();opt.threading.private_threadpool_size=2
    return d.batch(batch).with_options(opt).prefetch(1)
def predict(m,x,batch=128):
    return np.concatenate([np.asarray(m(x[i:i+batch],training=False)) for i in range(0,len(x),batch)])
def table(df,name):
    df.to_csv(ROOT/'results'/name,index=False);display(df)
print('Python',platform.python_version(),'TensorFlow',tf.__version__,'NumPy',np.__version__)

## 2. Small networks for two spirals

Use the TensorFlow Playground spiral equations, 250 points per arm, no noise. Remove the two identical origin samples carrying contradictory labels. Hold out 20% for testing; split the rest into 60% training / 20% validation overall. Both classes are stratified. No engineered features are allowed. All candidates have tanh hidden/output activations, labels −1/+1, MSE, full-batch Adam at 0.01, and 5,000 updates. We require **at least 95% validation accuracy in both seeds (42,43)**. Parameter count includes biases: a dense layer has `(input_width + 1) × output_width` parameters. This design isolates width/depth under a fixed budget, but neither exhausts architectures nor proves optimization has found every candidate's best solution.

The official Playground typically uses SGD and different sampling/splits; therefore this is a controlled Keras experiment with equivalent geometry, not a claim that its numerical settings reproduce browser training.

### TensorFlow Playground: observed architecture comparison

The browser experiments use the spiral dataset, raw X1/X2 only, 50% training / 50% test data, noise 0, batch size 10, learning rate 0.03, tanh, and no regularization. Each architecture was trained for approximately 5,000 epochs. The screenshots below record actual runs; losses are not classification accuracies.

| Hidden layers | Parameters (including biases) | Epoch | Training loss | Test loss |
| --- | ---: | ---: | ---: | ---: |
| 4 → 2 → 2 | 31 | 5030 | 0.329 | 0.418 |
| 6 → 6 | 67 | 5017 | 0.143 | 0.386 |
| 7 → 7 | 85 | 5000 | 0.009 | 0.178 |
| 8 → 8 | 105 | 5009 | 0.041 | 0.130 |

The 4–2–2 and 6–6 runs fail to follow substantial portions of the spiral. The 7–7 network captures the main spiral pattern with fewer parameters than 8–8, and achieves the lowest training loss, but its larger train/test gap and irregular boundaries indicate poorer generalization in this run. The 8–8 run has the lowest displayed test loss and is my preferred performance/size compromise among these browser runs, although local errors remain. None of these screenshots establishes perfect classification or a globally smallest solution. Initialization can affect optimization, and one run per architecture cannot separate architecture effects from random variation. The displayed test results are used for this exploratory comparison, so they are not an untouched final estimate after architecture selection.

The controlled Keras search below uses a separate split, optimizer, and explicit validation-accuracy criterion. Its accuracy results must not be attributed to these Playground screenshots. “Smallest” in that search means smallest among its tested candidates meeting its stated criterion.


[Figure omitted from source-only collection: Playground 4 → 2 → 2, epoch 5030]

[Figure omitted from source-only collection: Playground 6 → 6, epoch 5017]

[Figure omitted from source-only collection: Playground 7 → 7, epoch 5000 (test points shown)]

[Figure omitted from source-only collection: Playground 8 → 8, epoch 5009]

In [ ]:
r=np.arange(1,250)/250*5
xy=[]; labels=[]
for delta,label in [(0,-1.),(np.pi,1.)]:
    theta=1.75*np.arange(1,250)/250*2*np.pi+delta
    xy.extend(np.c_[r*np.sin(theta),r*np.cos(theta)]);labels.extend([label]*len(r))
X=np.asarray(xy,dtype='float32');y=np.asarray(labels,dtype='float32')[:,None]
ix=np.arange(len(y));dev,te=train_test_split(ix,test_size=.2,stratify=y[:,0],random_state=42)
tr,va=train_test_split(dev,test_size=.25,stratify=y[dev,0],random_state=42)
shapes=[(),(2,),(4,),(8,),(2,2),(3,3),(4,4),(6,4),(6,6),(4,4,4),(8,8),(8,8,8),(16,16)]
spiral_models={};rows=[]
for widths in shapes:
    for seed in [42,43]:
        keras.utils.set_random_seed(seed)
        m=keras.Sequential([keras.Input((2,))]+[keras.layers.Dense(w,activation='tanh') for w in widths]+[keras.layers.Dense(1,activation='tanh')])
        opt=keras.optimizers.Adam(.01)
        xx=tf.constant(X[tr]);yy=tf.constant(y[tr])
        @tf.function
        def step():
            with tf.GradientTape() as tape:loss=tf.reduce_mean(tf.square(m(xx)-yy))
            opt.apply_gradients(zip(tape.gradient(loss,m.trainable_variables),m.trainable_variables))
        for _ in range(5000):step()
        row={'architecture':str(widths),'seed':seed,'parameters':m.count_params(),'train_accuracy':np.mean((predict(m,X[tr])>0)==(y[tr]>0)),'val_accuracy':np.mean((predict(m,X[va])>0)==(y[va]>0))}
        rows.append(row);spiral_models[(str(widths),seed)]=m
spiral=pd.DataFrame(rows);table(spiral,'spiral_runs.csv')
summary=spiral.groupby('architecture').agg(parameters=('parameters','first'),minimum_val=('val_accuracy','min'),mean_val=('val_accuracy','mean')).reset_index().sort_values('parameters')
qualified=summary[summary.minimum_val>=.95]
assert len(qualified)>0,'No network satisfies the criterion; expand the search.'
chosen=qualified.iloc[0].architecture
seed=int(spiral[spiral.architecture==chosen].sort_values('val_accuracy',ascending=False).iloc[0].seed)
sm=spiral_models[(chosen,seed)]
print('Selected:',chosen,'seed',seed,'parameters',sm.count_params(),'held-out test accuracy',np.mean((predict(sm,X[te])>0)==(y[te]>0)))
table(summary,'spiral_summary.csv');sm.save(ROOT/'models/spiral.keras')

In [ ]:
# Each panel pairs a decision surface with the actual network connectivity.
import ast
alternatives=qualified.architecture.tolist()[:3]
fig,axes=plt.subplots(2,len(alternatives),figsize=(5*len(alternatives),8),squeeze=False)
gxx,gyy=np.meshgrid(np.linspace(-5.5,5.5,240),np.linspace(-5.5,5.5,240));grid=np.c_[gxx.ravel(),gyy.ravel()].astype('float32')
for col,shape in enumerate(alternatives):
    sd=int(spiral[spiral.architecture==shape].sort_values('val_accuracy',ascending=False).iloc[0].seed)
    model=spiral_models[(shape,sd)];ax=axes[0,col]
    zz=predict(model,grid).reshape(gxx.shape);ax.contourf(gxx,gyy,zz,levels=np.linspace(-1,1,21),cmap='coolwarm',alpha=.75)
    ax.scatter(X[tr,0],X[tr,1],c=y[tr,0],cmap='coolwarm',s=8,edgecolors='k',linewidths=.2)
    ax.set(title=f'{shape}: {model.count_params()} parameters',xlabel='x',ylabel='y');ax.set_aspect('equal')
    ax=axes[1,col];widths=[2,*ast.literal_eval(shape),1]
    points=[[(i,j-(w-1)/2) for j in range(w)] for i,w in enumerate(widths)]
    for left,right in zip(points[:-1],points[1:]):
        for a in left:
            for b in right:ax.plot([a[0],b[0]],[a[1],b[1]],c='gray',lw=.5,alpha=.4)
    for layer in points:
        a=np.array(layer);ax.scatter(a[:,0],a[:,1],s=80,c='#287caf',zorder=3)
    ax.set_title('Raw input → tanh hidden layers → tanh output');ax.axis('off')
plt.tight_layout();show('spiral_architectures.png')
print('Choice rationale: the first qualifying row has the lowest parameter count under the two-seed criterion. Smaller failures show a tradeoff, not mathematical impossibility. Additional seeds and longer training could change the result.')

## 3. Function approximation and extrapolation

Two different Keras FFNs: sine uses 32–32 tanh units; the second function `f(x)=0.4x²+sin(3x)` uses 64–64–32 tanh units. Outputs are linear. Input is divided by the training interval radius; no test information enters scaling. Draw 1,600 training-domain points, reserve 20% for validation, optimize MSE with Adam (0.003), at most 600 epochs, early stopping patience 45. Compare a dense interpolation grid and a separate extrapolation interval: (π,2π) for sine and (2,4) for the second function.

In [ ]:
function_rows=[]
for name,fn,radius,widths in [('sine',np.sin,np.pi,[32,32]),('quadratic_sine',lambda z:.4*z*z+np.sin(3*z),2.,[64,64,32])]:
    keras.utils.set_random_seed(42);rng=np.random.default_rng(42)
    xx=rng.uniform(-radius,radius,(1600,1)).astype('float32');yy=fn(xx).astype('float32')
    it,iv=train_test_split(np.arange(len(xx)),test_size=.2,random_state=42)
    model=keras.Sequential([keras.Input((1,)),keras.layers.Rescaling(1/radius)]+[keras.layers.Dense(w,activation='tanh') for w in widths]+[keras.layers.Dense(1)])
    model.compile(keras.optimizers.Adam(.003),loss='mse')
    h=model.fit(dataset(xx[it],yy[it],True),validation_data=dataset(xx[iv],yy[iv]),epochs=600,verbose=0,callbacks=[keras.callbacks.EarlyStopping(patience=45,restore_best_weights=True)])
    a=np.linspace(-radius,radius,600,dtype='float32')[:,None];b=np.linspace(radius,2*radius,600,dtype='float32')[:,None]
    pa=predict(model,a);pb=predict(model,b)
    function_rows.append(dict(function=name,hidden_layers=str(widths),parameters=model.count_params(),epochs=len(h.history['loss']),interpolation_mse=float(np.mean((pa-fn(a))**2)),extrapolation_mse=float(np.mean((pb-fn(b))**2))))
    fig,ax=plt.subplots(1,3,figsize=(14,3.5))
    for axis,z,pred,title in [(ax[0],a,pa,'Interpolation'),(ax[1],b,pb,'Extrapolation')]:
        axis.plot(z,fn(z),label='True function');axis.plot(z,pred,'--',label='FFN');axis.set(title=title,xlabel='x',ylabel='f(x)');axis.legend()
    ax[2].semilogy(h.history['loss'],label='Train');ax[2].semilogy(h.history['val_loss'],label='Validation');ax[2].set(title=name,xlabel='Epoch',ylabel='MSE');ax[2].legend()
    show(name+'.png');model.summary();model.save(ROOT/'models'/f'{name}.keras')
table(pd.DataFrame(function_rows),'function_results.csv')

**Interpretation.** Universal approximation is an existence result: an adequately wide network with a suitable nonlinearity can approximate a continuous function arbitrarily closely on a compact domain. It does not promise that a particular finite network, optimizer, sample, or training budget finds such weights, nor does it promise extrapolation. Low interpolation error supports practical approximation on the sampled domain. Large out-of-domain errors do not disprove the theorem. Tanh hidden units can saturate outside the observed range; neither periodicity nor unbounded quadratic growth is built into these models. A sine-specific periodic inductive bias could improve extrapolation, but would answer a different modeling question.

## 4a. Yoga-107 data audit and leakage control

Source: [Kaggle: Yoga Pose Image Classification Dataset](https://www.kaggle.com/datasets/shrutisaxena/yoga-pose-image-classification-dataset). Load all 107 directory labels. Decode and orient each image, convert to RGB, hash its dimensions plus decoded pixels, discard unreadable images, exclude exact duplicates with conflicting labels, and keep only one copy of same-label exact duplicates **before splitting**. Resize with aspect-preserving padding to 64×64, avoiding cropping limbs. GIF files use the first frame. Exact hashing does not detect near-duplicates or shared people; identity/session-aware evaluation would be stronger, but such metadata is unavailable.

The requested 80/20 train/test split is stratified. An inner 20% of the 80% development set is reserved for validation, giving 64/16/20 overall. Optimizer tuning, stopping, and model selection use only training/validation. Class proportions are retained; macro F1 gives equal weight to each pose.

In [ ]:
data_root=ROOT/'data/yoga107'
if not data_root.exists():
    import urllib.request
    archive=ROOT/'data/yoga107.zip'
    if not archive.exists():
        urllib.request.urlretrieve('https://www.kaggle.com/api/v1/datasets/download/shrutisaxena/yoga-pose-image-classification-dataset',archive)
    with zipfile.ZipFile(archive) as z:z.extractall(data_root)
paths=sorted(p for p in data_root.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.jpe','.gif','.bmp','.webp'})
records=[];thumbs={};bad=[]
for p in paths:
    try:
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im).convert('RGB');w,h=im.size
            digest=hashlib.sha256(str((w,h)).encode()+im.tobytes()).hexdigest()
            thumb=np.asarray(ImageOps.pad(im,(64,64),method=Image.Resampling.BILINEAR,color=(128,128,128)))
        records.append(dict(path=str(p.relative_to(data_root)),label=p.parent.name,width=w,height=h,digest=digest));thumbs[str(p.relative_to(data_root))]=thumb
    except Exception as e:bad.append({'path':str(p),'error':str(e)})
audit=pd.DataFrame(records);conflict_hash=audit.groupby('digest').label.nunique();conflict_hash=set(conflict_hash[conflict_hash>1].index)
conflicted=audit.digest.isin(conflict_hash)
clean=audit.loc[~conflicted].drop_duplicates('digest').copy().reset_index(drop=True)
classes=sorted(clean.label.unique());assert len(classes)==107
class_id={c:i for i,c in enumerate(classes)}
Xy=np.stack([thumbs[p] for p in clean.path]);Y=np.array([class_id[c] for c in clean.label],dtype='int32');del thumbs
idx=np.arange(len(Y));dev_idx,test_idx=train_test_split(idx,test_size=.2,random_state=42,stratify=Y)
train_idx,val_idx=train_test_split(dev_idx,test_size=.2,random_state=42,stratify=Y[dev_idx])
clean['split']='test';clean.loc[train_idx,'split']='train';clean.loc[val_idx,'split']='validation'
assert all(len(np.unique(Y[a]))==107 for a in [train_idx,val_idx,test_idx])
clean.to_csv(ROOT/'results/data_manifest.csv',index=False);audit.to_csv(ROOT/'results/data_audit.csv',index=False)
(ROOT/'models/classes.json').write_text(json.dumps(classes,indent=2))
eda={'original_files':len(paths),'decoded':len(audit),'unreadable':len(bad),'conflicting_label_images_removed':int(conflicted.sum()),'same_label_duplicates_removed':int((~conflicted).sum()-len(clean)),'clean_images':len(clean),'classes':len(classes),'training':len(train_idx),'validation':len(val_idx),'test':len(test_idx)}
(ROOT/'results/eda.json').write_text(json.dumps(eda,indent=2));print(eda)
counts=clean.label.value_counts();print('Class size:',counts.describe().to_dict())
fig,ax=plt.subplots(1,3,figsize=(14,3.5));ax[0].hist(counts,bins=15);ax[0].set(xlabel='Images per class',ylabel='Classes',title='Class imbalance')
ax[1].scatter(audit.width,audit.height,s=4,alpha=.4);ax[1].set(xscale='log',yscale='log',xlabel='Width',ylabel='Height',title='Native resolutions')
ax[2].hist(audit.width/audit.height,bins=40,range=(0,4));ax[2].set(xlabel='Aspect ratio',title='Image geometry');show('yoga_eda.png')
fig,axs=plt.subplots(3,5,figsize=(13,8))
for ax,i in zip(axs.ravel(),np.random.default_rng(42).choice(train_idx,15,replace=False)):
    ax.imshow(Xy[i]);ax.set_title(classes[Y[i]],fontsize=8);ax.axis('off')
show('yoga_examples.png')

## 4b–c. Controlled CNN experiments

**Basic:** three 3×3 same-padded ReLU convolutions (8,16,32 filters), each followed by 2×2 max pooling; flatten; Dense(64,ReLU); Dense(107,softmax). All basic runs use **vanilla SGD (momentum=0, no Nesterov)**; compare initial learning rates 0.01, 0.05, 0.10.

**Advanced:** six convolutions, two each at 16,32,64 filters; batch normalization after every convolution, then ReLU; pool after each pair; Dense(128,ReLU), dropout 0.35, softmax output. Convolution and hidden dense weights receive L2=0.0001. Convolution biases are disabled because BN supplies an offset. One advanced run uses the best basic learning rate and vanilla SGD to help isolate architectural changes. A second uses SGD momentum 0.9 at 0.03, explicitly distinguished from the required vanilla baseline.

All runs use batch 64, sparse cross entropy, early stopping on validation loss (patience 9), and ReduceLROnPlateau (factor 0.5, patience 4). Budgets: 35 basic / 45 advanced epochs. Restore the minimum-validation-loss epoch for each run, then rank restored models by validation macro F1. Train/test curves are reconstructed from saved epoch checkpoints **only after selection is frozen**. Test curves are reporting diagnostics, never stopping/tuning signals. TensorBoard receives epoch metrics and an explicit inference graph.

In [ ]:
def basic_model():
    m=keras.Sequential([keras.Input((64,64,3)),keras.layers.Rescaling(1/255.)],name='basic')
    for f in [8,16,32]:m.add(keras.layers.Conv2D(f,3,padding='same',activation='relu'));m.add(keras.layers.MaxPooling2D())
    m.add(keras.layers.Flatten());m.add(keras.layers.Dense(64,activation='relu'));m.add(keras.layers.Dense(107,activation='softmax'))
    return m
def advanced_model():
    reg=keras.regularizers.L2(1e-4)
    m=keras.Sequential([keras.Input((64,64,3)),keras.layers.Rescaling(1/255.)],name='advanced')
    for f in [16,32,64]:
        for _ in range(2):
            m.add(keras.layers.Conv2D(f,3,padding='same',use_bias=False,kernel_regularizer=reg));m.add(keras.layers.BatchNormalization());m.add(keras.layers.Activation('relu'))
        m.add(keras.layers.MaxPooling2D())
    m.add(keras.layers.Flatten());m.add(keras.layers.Dense(128,activation='relu',kernel_regularizer=reg));m.add(keras.layers.Dropout(.35));m.add(keras.layers.Dense(107,activation='softmax'))
    return m
train_ds=dataset(Xy[train_idx],Y[train_idx],True);val_ds=dataset(Xy[val_idx],Y[val_idx])
experiments=[];models={};histories={}
def train_run(name,factory,lr,momentum,epochs):
    keras.utils.set_random_seed(42);m=factory();m.compile(keras.optimizers.SGD(learning_rate=lr,momentum=momentum,nesterov=False),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    cp=ROOT/'checkpoints'/name;cp.mkdir(exist_ok=True)
    for p in cp.glob('*.weights.h5'):p.unlink()
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=9,restore_best_weights=True),keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=4,min_lr=1e-5),keras.callbacks.ModelCheckpoint(str(cp/'{epoch:03d}.weights.h5'),save_weights_only=True),keras.callbacks.CSVLogger(str(ROOT/'results'/f'{name}_history.csv')),keras.callbacks.TensorBoard(log_dir=str(ROOT/'logs'/name),histogram_freq=0,write_graph=True)]
    t=time.perf_counter();h=m.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=0)
    elapsed=time.perf_counter()-t;best_epoch=int(np.argmin(h.history['val_loss'])+1);m.load_weights(cp/f'{best_epoch:03d}.weights.h5')
    probs=predict(m,Xy[val_idx]);pr=probs.argmax(1)
    row=dict(run=name,initial_lr=lr,momentum=momentum,parameters=m.count_params(),epochs=len(h.history['loss']),best_epoch=best_epoch,seconds=elapsed,val_accuracy=accuracy_score(Y[val_idx],pr),val_macro_f1=f1_score(Y[val_idx],pr,average='macro',zero_division=0),val_cross_entropy=log_loss(Y[val_idx],probs,labels=np.arange(107)))
    experiments.append(row);models[name]=m;histories[name]=h.history;m.save(ROOT/'models'/f'{name}.keras')
    writer=tf.summary.create_file_writer(str(ROOT/'logs'/name/'inference_graph'))
    @tf.function(input_signature=[tf.TensorSpec([None,64,64,3],tf.float32)])
    def inference(z):return m(z,training=False)
    with writer.as_default():tf.summary.graph(inference.get_concrete_function().graph)
    writer.flush();writer.close();print(row,flush=True)
    pd.DataFrame(experiments).to_csv(ROOT/'results/validation_comparison.csv',index=False)
for lr in [.01,.05,.1]:train_run(f'basic_sgd_{lr}',basic_model,lr,0.,35)
best_basic=max(experiments,key=lambda q:(q['val_macro_f1'],-q['val_cross_entropy']))
train_run('advanced_vanilla',advanced_model,best_basic['initial_lr'],0.,45)
train_run('advanced_momentum',advanced_model,.03,.9,45)
ranking=pd.DataFrame(experiments).sort_values(['val_macro_f1','val_cross_entropy'],ascending=[False,True])
best_name=ranking.iloc[0]['run'];best=models[best_name]
selection={'selected':best_name,'criterion':'maximum validation macro F1; tie minimum validation cross entropy','test_used_for_selection':False}
(ROOT/'results/frozen_selection.json').write_text(json.dumps(selection,indent=2));best.save(ROOT/'models/best.keras')
table(ranking,'validation_comparison.csv');print('Frozen choice:',best_name)

In [ ]:
# All test access below occurs after the fixed selection above.
comparison=[];test_predictions={}
for row in experiments:
    name=row['run'];p=predict(models[name],Xy[test_idx]);test_predictions[name]=p
    comparison.append({**row,'test_accuracy':accuracy_score(Y[test_idx],p.argmax(1)),'test_macro_f1':f1_score(Y[test_idx],p.argmax(1),average='macro',zero_division=0),'test_weighted_f1':f1_score(Y[test_idx],p.argmax(1),average='weighted',zero_division=0),'test_cross_entropy':log_loss(Y[test_idx],p,labels=np.arange(107))})
comparison=pd.DataFrame(comparison);table(comparison,'test_comparison.csv')
fig,ax=plt.subplots(1,2,figsize=(13,4));comparison.plot.bar(x='run',y=['val_accuracy','test_accuracy','test_macro_f1'],ax=ax[0]);comparison.plot.bar(x='run',y=['val_cross_entropy','test_cross_entropy'],ax=ax[1]);ax[0].set_ylabel('Score');ax[1].set_ylabel('Predictive cross entropy');show('experiment_comparison.png')
for name,h in histories.items():
    fig,ax=plt.subplots(1,2,figsize=(11,3));ax[0].plot(h['accuracy'],label='Train');ax[0].plot(h['val_accuracy'],label='Validation');ax[1].plot(h['loss'],label='Train objective');ax[1].plot(h['val_loss'],label='Validation objective')
    for a in ax:a.set_xlabel('Epoch');a.legend()
    ax[0].set_title(name+' accuracy');ax[1].set_title(name+' loss (includes L2 where used)');show(name+'_convergence.png')
# The following test traces do not influence any selection decision.
advanced_name=ranking[ranking.run.str.startswith('advanced')].iloc[0]['run']
for name in [best_basic['run'],advanced_name]:
    m=models[name];held=m.get_weights();trace=[]
    for p in sorted((ROOT/'checkpoints'/name).glob('*.weights.h5')):
        m.load_weights(p);prob=predict(m,Xy[test_idx]);reg=float(tf.add_n(m.losses)) if m.losses else 0.
        trace.append({'epoch':int(p.name.split('.')[0]),'test_accuracy':accuracy_score(Y[test_idx],prob.argmax(1)),'test_cross_entropy':log_loss(Y[test_idx],prob,labels=np.arange(107)),'test_objective':log_loss(Y[test_idx],prob,labels=np.arange(107))+reg})
    m.set_weights(held);trace=pd.DataFrame(trace);trace.to_csv(ROOT/'results'/f'{name}_test_trace.csv',index=False)
    h=histories[name];fig,ax=plt.subplots(1,2,figsize=(11,3.5));ax[0].plot(np.arange(1,len(h['accuracy'])+1),h['accuracy'],label='Train');ax[0].plot(trace.epoch,trace.test_accuracy,label='Test (post hoc)');ax[1].plot(np.arange(1,len(h['loss'])+1),h['loss'],label='Train objective');ax[1].plot(trace.epoch,trace.test_objective,label='Test objective (post hoc)')
    for a in ax:a.set_xlabel('Epoch');a.legend()
    ax[0].set_title(name+' accuracy');ax[1].set_title(name+' objective');show(name+'_train_test.png')

In [ ]:
p=test_predictions[best_name];yp=p.argmax(1);cm=confusion_matrix(Y[test_idx],yp,labels=np.arange(107))
pd.DataFrame(cm,index=classes,columns=classes).to_csv(ROOT/'results/confusion_matrix.csv')
report=pd.DataFrame(classification_report(Y[test_idx],yp,labels=np.arange(107),target_names=classes,output_dict=True,zero_division=0)).T
report.to_csv(ROOT/'results/classification_report.csv');worst=report.loc[classes].sort_values(['f1-score','recall']).head(12);display(worst)
fig,ax=plt.subplots(figsize=(13,11));im=ax.imshow(cm,cmap='Blues');ax.set(xlabel='Predicted class index (see classes.json)',ylabel='True class index',title=f'{best_name}: all 107 classes');fig.colorbar(im,ax=ax);show('confusion_full.png')
pairs=[(int(cm[i,j]),classes[i],classes[j]) for i in range(107) for j in range(107) if i!=j and cm[i,j]>0];pairs=sorted(pairs,reverse=True)[:12];table(pd.DataFrame(pairs,columns=['count','true_pose','predicted_pose']),'top_confusions.csv')
errors=np.flatnonzero(yp!=Y[test_idx]);fig,axs=plt.subplots(3,4,figsize=(14,10))
for ax,j in zip(axs.ravel(),errors[:12]):
    ax.imshow(Xy[test_idx[j]]);ax.set_title(f'True: {classes[Y[test_idx[j]]]}\nPred: {classes[yp[j]]}',fontsize=8);ax.axis('off')
for ax in axs.ravel()[min(12,len(errors)):]:ax.axis('off')
show('misclassified_examples.png')
print('Lowest-F1 poses:',', '.join(worst.index[:6]))
print('These are descriptive test findings, not proof of anatomical similarity. Review the images and support counts; per-class estimates based on small test sets are uncertain. A pose can have zero F1 because it is never predicted.')

## 4d. Save and quantize the chosen model

Convert the same validation-selected model to two TensorFlow Lite files: float32 control and float16 weight quantization. Evaluate both on **every identical held-out test image**, with float32 raw RGB input, and record file size, accuracy, macro F1, probability differences, prediction agreement, and CPU median/p95 latency. Float16 stores weights more compactly; CPU kernels may upcast them, so smaller files do not imply faster CPU inference or half runtime memory. Latency here is a one-thread CPU microbenchmark, not a mobile-device result. Loading/decompression time is excluded. No calibration examples are needed for this weight-only scheme.

In [ ]:
export_dir=ROOT/'models/exported_best'
@tf.function(input_signature=[tf.TensorSpec([1,64,64,3],tf.float32,name='images')])
def serve(images):return {'probabilities':best(images,training=False)}
# Freeze resource variables explicitly so converter output contains the trained constants.
# This also provides a numerical control before quantization.
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
frozen = convert_variables_to_constants_v2(serve.get_concrete_function())
control = np.asarray(frozen(tf.constant(Xy[test_idx[:1]],dtype=tf.float32))[0])
np.testing.assert_allclose(control, predict(best,Xy[test_idx[:1]]), rtol=1e-4, atol=1e-5)
import shutil
if export_dir.exists(): shutil.rmtree(export_dir)
module = tf.Module()
module.inference = frozen
tf.saved_model.save(module, str(export_dir), signatures={'serving_default': frozen})
quant_rows=[]
for kind in ['float32','float16']:
    converter=tf.lite.TFLiteConverter.from_concrete_functions([frozen])
    if kind=='float16':converter.optimizations=[tf.lite.Optimize.DEFAULT];converter.target_spec.supported_types=[tf.float16]
    blob=converter.convert();path=ROOT/'models'/f'best_{kind}.tflite';path.write_bytes(blob)
    interpreter=tf.lite.Interpreter(model_content=blob,num_threads=1);interpreter.allocate_tensors();inp=interpreter.get_input_details()[0];out=interpreter.get_output_details()[0]
    def infer_one(z):
        interpreter.set_tensor(inp['index'],z[None].astype('float32'));interpreter.invoke();return interpreter.get_tensor(out['index'])[0].copy()
    prob=np.stack([infer_one(z) for z in Xy[test_idx]])
    assert np.isfinite(prob).all(), 'Export produced nonfinite probabilities'
    np.testing.assert_allclose(prob.sum(axis=1), 1., atol=1e-5)
    if kind == 'float32':
        np.testing.assert_allclose(prob,p,atol=2e-4,rtol=2e-3)
        assert np.mean(prob.argmax(1)==yp) > .999, 'Float32 conversion fails the Keras control'
    assert len(blob)>1_000_000, 'Unexpectedly small model; trained constants may be missing' 
    for i in range(10):infer_one(Xy[test_idx[i]])
    times=[]
    for i in range(100):
        t=time.perf_counter();infer_one(Xy[test_idx[i%len(test_idx)]]);times.append((time.perf_counter()-t)*1000)
    quant_rows.append({'format':kind,'bytes':len(blob),'test_accuracy':accuracy_score(Y[test_idx],prob.argmax(1)),'test_macro_f1':f1_score(Y[test_idx],prob.argmax(1),average='macro',zero_division=0),'agreement_with_keras':np.mean(prob.argmax(1)==yp),'max_probability_difference':float(np.max(np.abs(prob-p))),'median_ms':np.median(times),'p95_ms':np.percentile(times,95)})
    print(kind,'tensor dtypes',pd.Series([str(d['dtype']) for d in interpreter.get_tensor_details()]).value_counts().to_dict())
quant=pd.DataFrame(quant_rows);table(quant,'quantization.csv')
print('Float16/float32 file-size ratio:',quant.iloc[1].bytes/quant.iloc[0].bytes)
fig,ax=plt.subplots(1,3,figsize=(12,3.5));ax[0].bar(quant['format'],quant.bytes/1024);ax[0].set_ylabel('File size (KiB)');ax[1].bar(quant['format'],quant.test_macro_f1);ax[1].set_ylabel('Test macro F1');ax[2].bar(quant['format'],quant.median_ms);ax[2].set_ylabel('Median CPU latency (ms)');show('quantization.png')

## 4e. Manual parameter calculation

For a 3×3 convolution: `9 × input_channels × output_channels + output_channels` with bias; omit the final term when bias is disabled. Dense: `input_units × output_units + output_units`. Batch normalization has two trainable vectors (gamma/beta) plus two nontrainable vectors (moving mean/variance): `4 × channels` total. Pooling, activation, rescaling, flattening, and dropout have zero parameters.

Basic: after three pools, 64×64 becomes 8×8. Conv counts are 224, 1,168, 4,640. Dense counts are `(8×8×32+1)×64 = 131,136` and `(64+1)×107 = 6,955`, totaling **144,123**, all trainable.

Advanced: convolution counts are 432, 2,304, 4,608, 9,216, 18,432, 36,864. BN total is `4×(16+16+32+32+64+64)=896`, of which 448 is trainable. Dense counts are `(8×8×64+1)×128=524,416` and `(128+1)×107=13,803`. Total **610,971**; **610,523 trainable**, **448 nontrainable**. The arithmetic is checked against Keras below; optimizer slots are not model parameters.

In [ ]:
for name,factory in [('basic',basic_model),('advanced',advanced_model)]:
    m=factory();m.summary();rows=[]
    for layer in m.layers:
        manual=0;trainable=0
        if isinstance(layer,keras.layers.Conv2D):
            kh,kw,ci,co=layer.kernel.shape;manual=int(kh*kw*ci*co+(co if layer.use_bias else 0));trainable=manual
        elif isinstance(layer,keras.layers.Dense):
            ci,co=layer.kernel.shape;manual=int(ci*co+(co if layer.use_bias else 0));trainable=manual
        elif isinstance(layer,keras.layers.BatchNormalization):
            c=int(layer.gamma.shape[0]);manual=4*c;trainable=2*c
        rows.append({'layer':layer.name,'manual_total':manual,'manual_trainable':trainable,'keras_total':layer.count_params()})
    d=pd.DataFrame(rows);assert (d.manual_total==d.keras_total).all();assert d.manual_total.sum()==m.count_params()
    assert d.manual_trainable.sum()==sum(np.prod(v.shape) for v in m.trainable_weights)
    table(d,name+'_manual_parameters.csv');print(name,'total',int(d.manual_total.sum()),'trainable',int(d.manual_trainable.sum()))

## Measured conclusions and TensorBoard evidence

The tables below are generated from this execution. Validation ranking chooses the deployment candidate even if another model has a better test score. Increasing capacity and adding regularization do not guarantee improvement: the advanced network also needs a suitable learning rate and adequate optimization. Training objective contains L2 for the advanced model, so use predictive cross entropy for clean between-model loss comparisons. These are single-split CNN experiments; uncertainty across seeds is not measured. Accuracy alone is insufficient for 107 unequal classes, hence macro F1 and the class-level error audit.

In [ ]:
base_row=comparison[comparison.run==best_basic['run']].iloc[0];adv_row=comparison[comparison.run==advanced_name].iloc[0];win=comparison[comparison.run==best_name].iloc[0]
print(f"Best basic: {base_row['run']}; test accuracy {base_row.test_accuracy:.4f}, macro F1 {base_row.test_macro_f1:.4f}.")
print(f"Best advanced by validation: {advanced_name}; test accuracy {adv_row.test_accuracy:.4f}, macro F1 {adv_row.test_macro_f1:.4f}.")
print(f"Advanced minus basic: accuracy {adv_row.test_accuracy-base_row.test_accuracy:+.4f}; macro F1 {adv_row.test_macro_f1-base_row.test_macro_f1:+.4f}.")
print(f"Saved model: {best_name}; test accuracy {win.test_accuracy:.4f}; macro F1 {win.test_macro_f1:.4f}.")
print(f"Quantization: {quant.iloc[0].bytes:,} → {quant.iloc[1].bytes:,} bytes; macro F1 change {quant.iloc[1].test_macro_f1-quant.iloc[0].test_macro_f1:+.6f}; CPU median {quant.iloc[0].median_ms:.3f} → {quant.iloc[1].median_ms:.3f} ms.")
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
tb=[]
for directory in sorted((ROOT/'logs').glob('*/*')):
    if list(directory.glob('events.out.tfevents.*')):
        ea=EventAccumulator(str(directory));ea.Reload();tags=ea.Tags();tb.append({'directory':str(directory.relative_to(ROOT)),'scalar_or_tensor_tags':str(tags['scalars']+tags['tensors']),'graph_present':tags['graph']})
table(pd.DataFrame(tb),'tensorboard_evidence.csv')
print('Inspect interactively: tensorboard --logdir logs; in Jupyter: %load_ext tensorboard then %tensorboard --logdir logs')
print('No TensorBoard UI screenshot is claimed; event files contain the logged graph and metrics.')

In [ ]:
# Re-evaluate the validation-selected basic/advanced models for class-level comparison.
# This is post-hoc reporting, after all model choices are frozen.
for name in [best_basic['run'], advanced_name]:
    probabilities = predict(models[name], Xy[test_idx])
    predictions = probabilities.argmax(1)
    per_class = pd.DataFrame(classification_report(
        Y[test_idx], predictions, labels=np.arange(107), target_names=classes,
        output_dict=True, zero_division=0)).T.loc[classes]
    per_class.to_csv(ROOT/'results'/f'{name}_per_class.csv')
    display(per_class.sort_values(['f1-score','recall']).head(8))

# Inspect the recorded TensorBoard inference graph, including function bodies.
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from collections import Counter
graph_events = EventAccumulator(str(ROOT/'logs'/best_name/'inference_graph'))
graph_events.Reload()
graph = graph_events.Graph()
operations = Counter(node.op for node in graph.node)
for function in graph.library.function:
    operations.update(node.op for node in function.node_def)
display(pd.DataFrame(operations.most_common(), columns=['TensorBoard graph operation','count']))
print('The inference graph contains', operations['Conv2D'], 'convolution operations and', operations['MatMul'], 'dense matrix multiplications.')
print('This graph records inference: batch normalization uses stored statistics and dropout is inactive. The epoch metrics and learning-rate changes are recorded in the train/validation event directories.')


## Final experiment interpretation

The best vanilla-SGD baseline by validation macro F1 is **basic_sgd_0.05**. Its held-out accuracy is **32.90%** and macro F1 is **0.2989**. The best advanced variant by the same criterion is **advanced_vanilla**, with test accuracy **49.01%** and macro F1 **0.4606**. Advanced minus basic is **+16.11 percentage points** in accuracy and **+0.1617** in macro F1. This is an empirical comparison of two configurations; multiple simultaneous architecture changes prevent a separate causal conclusion for BN, dropout, or L2.

The validation-selected deployment file is **advanced_vanilla**. Its test accuracy is **49.01%**, leaving a substantial fraction of images misclassified. The results do not support treating this modest custom model as a reliable deployed yoga coach. Resolution is restricted to 64×64, only 3,712 examples are used for gradient updates, and related poses can differ by small limb arrangements. These are plausible limitations, not experimentally isolated causes. Further work should prioritize reviewing labels and near-duplicates, higher-resolution inputs, more data, transfer learning as an additional comparison, and group splits by person/source when metadata becomes available.

For the same advanced architecture, vanilla SGD achieves validation macro F1 **0.4710**, whereas the momentum configuration achieves **0.3787**. Their test macro F1 scores are **0.4606** and **0.3912**, respectively. The initial learning rate also changes between these runs, so the difference is a comparison of optimizer configurations, not an isolated causal effect of momentum. The advanced model has **610,971 parameters**, about **4.24 times** the basic model's 144,123; improvement must be weighed against increased storage and computation.

### Poses the basic model struggles with

| pose | precision | recall | f1-score | support |
| --- | --- | --- | --- | --- |
| agnistambhasana | 0.0000 | 0.0000 | 0.0000 | 7.0000 |
| ardha bhekasana | 0.0000 | 0.0000 | 0.0000 | 8.0000 |
| baddha konasana | 0.0000 | 0.0000 | 0.0000 | 14.0000 |
| bhekasana | 0.0000 | 0.0000 | 0.0000 | 7.0000 |
| eka pada koundinyanasana i | 0.0000 | 0.0000 | 0.0000 | 7.0000 |
| eka pada koundinyanasana ii | 0.0000 | 0.0000 | 0.0000 | 8.0000 |
| eka pada rajakapotasana ii | 0.0000 | 0.0000 | 0.0000 | 9.0000 |
| ganda bherundasana | 0.0000 | 0.0000 | 0.0000 | 8.0000 |

These are the eight lowest-F1 classes for the validation-selected basic model, with support shown to avoid overinterpreting small test samples. A zero F1 can reflect never predicting a class, rather than an inability to detect every example of it. The corresponding predictions and full confusion matrix are saved under `results/`.

### Poses the advanced model struggles with

| pose | precision | recall | f1-score | support |
| --- | --- | --- | --- | --- |
| eka pada koundinyanasana i | 0.0000 | 0.0000 | 0.0000 | 7.0000 |
| eka pada koundinyanasana ii | 0.0000 | 0.0000 | 0.0000 | 8.0000 |
| marichyasana iii | 0.0000 | 0.0000 | 0.0000 | 6.0000 |
| padangusthasana | 0.0000 | 0.0000 | 0.0000 | 4.0000 |
| tulasana | 0.0000 | 0.0000 | 0.0000 | 5.0000 |
| supta matsyendrasana | 0.0833 | 0.0909 | 0.0870 | 11.0000 |
| eka pada rajakapotasana ii | 0.1429 | 0.1111 | 0.1250 | 9.0000 |
| ardha bhekasana | 0.1667 | 0.1250 | 0.1429 | 8.0000 |

Comparing these tables can reveal whether added capacity helps particular categories even when the aggregate score changes little. Inspect the actual images and confusion-pair plots before attributing mistakes to anatomical similarity.

### Quantization finding

The float32 TFLite file is **2,446,396 bytes** and the float16-weight file is **1,227,696 bytes**, a **49.8% reduction**. Macro F1 changes by **+0.000105**. Median one-thread CPU latency changes from **1.074 ms** to **1.071 ms**. Those measurements support a storage/accuracy tradeoff for these files; device-level speed, memory, and energy still require testing on the intended phone.


[Figure omitted from source-only collection: basic_sgd_0.05 confusion pairs]

[Figure omitted from source-only collection: advanced_vanilla confusion pairs]